In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

os.environ.setdefault("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "minio")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "minio123")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Dota 2 EDA")

2026/10/15 13:42:29 INFO mlflow.tracking.fluent: Experiment with name 'Dota 2 EDA' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://mlflow/2', creation_time=1792060949522, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1792060949522, lifecycle_stage='active', name='Dota 2 EDA', tags={}, trace_location=None, workspace='default'>

In [2]:
df = pd.read_csv("data/matches.csv")
print("Размер датасета:", df.shape)
print("\nПервые 5 строк:")
display(df.head())
print("\nОписательная статистика:")
display(df.describe())
print("\nПропуски:")
print(df.isna().sum())

Размер датасета: (100, 8)

Первые 5 строк:


,match_id,duration,radiant_win,radiant_score,dire_score,game_mode,lobby_type,first_blood_time
0,9045858554,0,0,0,0,13,14,0
1,9045858415,0,0,0,0,1,4,0
2,9045857283,0,0,0,0,1,4,0
3,9045856415,0,0,0,0,13,14,0
4,9045855073,0,0,0,0,1,4,0



Описательная статистика:


,match_id,duration,radiant_win,radiant_score,dire_score,game_mode,lobby_type,first_blood_time
count,1.000000e+02,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
mean,9.045846e+09,704.770000,0.480000,17.160000,14.600000,17.790000,1.660000,33.280000
std,4.040236e+03,404.656928,0.502117,14.780358,13.817717,8.836672,3.294378,47.224818
min,9.045842e+09,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
25%,9.045843e+09,434.000000,0.000000,2.750000,4.750000,13.000000,0.000000,0.000000
50%,9.045844e+09,848.000000,0.000000,15.000000,11.500000,23.000000,0.000000,7.500000
75%,9.045848e+09,1020.750000,1.000000,31.000000,21.000000,23.000000,4.000000,61.000000
max,9.045859e+09,1203.000000,1.000000,49.000000,64.000000,23.000000,14.000000,225.000000



Пропуски:
match_id            0
duration            0
radiant_win         0
radiant_score       0
dire_score          0
game_mode           0
lobby_type          0
first_blood_time    0
dtype: int64


In [3]:
with mlflow.start_run(run_name="EDA Analysis"):
    mlflow.log_param("dataset_size", len(df))
    mlflow.log_param("columns", ", ".join(df.columns))

    # 1 Распределение длительности матчей 
    fig, ax = plt.subplots()
    sns.histplot(df["duration"] / 60, bins=30, ax=ax, kde=True)
    ax.set_title("Распределение длительности матчей")
    ax.set_xlabel("Длительность (минуты)")
    ax.set_ylabel("Количество матчей")
    fig.savefig("duration_dist.png", bbox_inches="tight")
    mlflow.log_artifact("duration_dist.png")
    plt.close(fig)

    # 2 Победители Radiant vs Dire 
    fig, ax = plt.subplots(figsize=(6, 6))
    counts = df["radiant_win"].value_counts()
    ax.pie(
        counts,
        labels=["Dire Win", "Radiant Win"],
        autopct="%1.1f%%",
        colors=["#d9534f", "#5cb85c"],
        startangle=90,
    )
    ax.set_title("Соотношение побед Radiant и Dire")
    fig.savefig("win_rate.png", bbox_inches="tight")
    mlflow.log_artifact("win_rate.png")
    plt.close(fig)

    # 3 Распределение счёта команд 
    fig, ax = plt.subplots()
    sns.histplot(df["radiant_score"], bins=20, ax=ax, label="Radiant", color="#5cb85c", alpha=0.6)
    sns.histplot(df["dire_score"], bins=20, ax=ax, label="Dire", color="#d9534f", alpha=0.6)
    ax.set_title("Распределение счёта команд")
    ax.set_xlabel("Убийства (score)")
    ax.legend()
    fig.savefig("score_dist.png", bbox_inches="tight")
    mlflow.log_artifact("score_dist.png")
    plt.close(fig)

    # 4 Корреляционная матрица 
    numeric_cols = ["duration", "radiant_win", "radiant_score", "dire_score", "first_blood_time"]
    corr = df[numeric_cols].corr()
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", ax=ax, square=True)
    ax.set_title("Корреляционная матрица признаков")
    fig.savefig("correlation.png", bbox_inches="tight")
    mlflow.log_artifact("correlation.png")
    plt.close(fig)

    # 5 Метрики 
    mlflow.log_metric("avg_duration_min", df["duration"].mean() / 60)
    mlflow.log_metric("median_duration_min", df["duration"].median() / 60)
    mlflow.log_metric("radiant_win_rate", df["radiant_win"].mean())
    mlflow.log_metric("avg_radiant_score", df["radiant_score"].mean())
    mlflow.log_metric("avg_dire_score", df["dire_score"].mean())

    print("EDA завершён, артефакты залогированы в MLflow")

EDA завершён, артефакты залогированы в MLflow
🏃 View run EDA Analysis at: http://localhost:5000/#/experiments/2/runs/39ab9b7f50574d7a8fa1b5476cf982b1
🧪 View experiment at: http://localhost:5000/#/experiments/2


In [4]:
print(f"""
- Всего матчей: {len(df)}
- Средняя длительность: {df['duration'].mean() / 60:.1f} мин
- Процент побед Radiant: {df['radiant_win'].mean() * 100:.1f}%
- Средний счёт Radiant: {df['radiant_score'].mean():.1f}
- Средний счёт Dire: {df['dire_score'].mean():.1f}
- Самая частая игра: game_mode={df['game_mode'].mode()[0] if not df['game_mode'].isna().all() else 'N/A'}
""")


- Всего матчей: 100
- Средняя длительность: 11.7 мин
- Процент побед Radiant: 48.0%
- Средний счёт Radiant: 17.2
- Средний счёт Dire: 14.6
- Самая частая игра: game_mode=23

